# PanneauNet — Belgian Traffic Sign Recognition


This notebook is Part 1 of a tutorial that trains a deep learning model to recognize Belgian traffic signs. **PanneauNet** is a renamed, modernized, and bug-fixed version of the original *Traffic Signs Recognition with TensorFlow* tutorial by Waleed Abdulla (source: https://github.com/waleedka/traffic-signs-tensorflow). The original notebook was written for Python 3.5 and TensorFlow 0.11, both of which are long deprecated, so several parts of the code no longer run on modern environments. This version fixes those issues and ports the model to TensorFlow 2 / Keras. See the project README for the full list of fixes.

## First Objective: Traffic Sign Classification

We'll start with a simple goal: classification. Given an image of a traffic sign, our model should be able to tell its type (e.g. Stop sign, speed limit, yield sign, ...etc.). We'll work with images that are properly cropped such that the traffic sign takes most of the image.

This project uses Python 3.10+, TensorFlow 2.x, NumPy, scikit-image, and Matplotlib. A Dockerfile is included that bundles all of these in one place. You can run it with:

```
docker build -t panneaunet .
docker run -it -p 8888:8888 -v $(pwd):/panneaunet panneaunet
```

First step, let's import the libraries we need.


In [ ]:
import os
import random
import skimage.io      # FIX: skimage.data.imread was removed in modern scikit-image;
                        #      skimage.io.imread is the correct, still-supported function.
import skimage.transform
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# Allow image embedding in notebook
%matplotlib inline

print("TensorFlow version:", tf.__version__)


## Training Dataset

We're using the **Belgian Traffic Sign Dataset (BelgiumTSC)**. Go to http://btsd.ethz.ch/shareddata/ and download the training and test data. There are several datasets on that page, but you only need the two files listed under **"BelgiumTS for Classification (cropped images)"**:

* BelgiumTSC_Training (171.3 MBytes)
* BelgiumTSC_Testing (76.5 MBytes)

After downloading and expanding the files, your directory structure should look something like this:

```
datasets/BelgiumTS/Training/
datasets/BelgiumTS/Testing/
```

Each of the two directories above has 62 sub-directories named sequentially from 00000 to 00061 (**FIX:** the original tutorial text said "00000 to 00062", which is off by one — there are 62 directories in total, numbered 0 through 61). The directory name represents the label, and the images inside it are examples of that label.


## Parse and Load the Training Data

The **Training** directory contains sub-directories with sequential numerical names from 00000 to 00061. The name of the directory represents the labels from 0 to 61, and the images in each directory represent the traffic signs that belong to that label. The images are saved in the not-so-common .ppm format, which is supported by the scikit-image library.


In [ ]:
def load_data(data_dir):
    """Loads a data set and returns two lists:

    images: a list of Numpy arrays, each representing an image.
    labels: a list of numbers that represent the images labels.
    """
    # Get all subdirectories of data_dir. Each represents a label.
    directories = [d for d in os.listdir(data_dir)
                   if os.path.isdir(os.path.join(data_dir, d))]
    # Loop through the label directories and collect the data in
    # two lists, labels and images.
    labels = []
    images = []
    for d in directories:
        label_dir = os.path.join(data_dir, d)
        file_names = [os.path.join(label_dir, f)
                      for f in os.listdir(label_dir) if f.endswith(".ppm")]
        # For each label, load its images and add them to the images list.
        # And add the label number (i.e. directory name) to the labels list.
        for f in file_names:
            # FIX: skimage.data.imread(...) -> skimage.io.imread(...)
            images.append(skimage.io.imread(f))
            labels.append(int(d))
    return images, labels


# Load training and testing datasets.
# FIX: the dataset root is now read from the DATA_ROOT environment variable
# (defaulting to a local ./datasets folder) instead of a hardcoded /traffic
# path, so the notebook also works outside of the Docker container.
ROOT_PATH = os.environ.get("DATA_ROOT", "./datasets")
train_data_dir = os.path.join(ROOT_PATH, "BelgiumTS/Training")
test_data_dir = os.path.join(ROOT_PATH, "BelgiumTS/Testing")

images, labels = load_data(train_data_dir)


Here we're loading two lists:
* **images** a list of images, each image is represented by a numpy array.
* **labels** a list of labels. Integers with values between 0 and 61.

It's not usually a good idea to load the whole dataset into memory, but this dataset is small and we're trying to keep the code simple, so it's okay for now. For larger datasets, you'd want a separate pipeline (e.g. `tf.data.Dataset`) that streams batches from disk.


## Explore the Dataset

How many images and labels do we have?


In [ ]:
print("Unique Labels: {0}\nTotal Images: {1}".format(len(set(labels)), len(images)))


Display the first image of each label.


In [ ]:
def display_images_and_labels(images, labels):
    """Display the first image of each label."""
    unique_labels = set(labels)
    plt.figure(figsize=(15, 15))
    i = 1
    for label in unique_labels:
        # Pick the first image for each label.
        image = images[labels.index(label)]
        plt.subplot(8, 8, i)  # A grid of 8 rows x 8 columns
        plt.axis('off')
        plt.title("Label {0} ({1})".format(label, labels.count(label)))
        i += 1
        _ = plt.imshow(image)
    plt.show()

display_images_and_labels(images, labels)


That looks great! The traffic signs occupy most of the area of each image, which is going to make our job easier: we don't have to look for the sign in the image. And we have a variety of angles and lighting conditions, which will help our model generalize.

However, although the images are square-ish, they're not all the same size. They have different aspect ratios. Our network takes a fixed-size input, so we have a bit of pre-processing to do. We'll get to that soon, but first let's pick a label and see more of its images. Let's pick label 32:


In [ ]:
def display_label_images(images, label):
    """Display images of a specific label."""
    limit = 24  # show a max of 24 images
    plt.figure(figsize=(15, 5))
    i = 1

    start = labels.index(label)
    end = start + labels.count(label)
    for image in images[start:end][:limit]:
        plt.subplot(3, 8, i)  # 3 rows, 8 per row
        plt.axis('off')
        i += 1
        plt.imshow(image)
    plt.show()

display_label_images(images, 32)


Interesting! It looks like our dataset considers all speed limit signs to be of the same class regardless of the numbers on them. That's fine, as long as we know about it beforehand. I'll leave exploring other labels as an exercise for you — edit the code above and check other labels. Make sure to check labels 26 and 27. They also have numbers in a red circle, so our model will have to get really good to differentiate between these three classes.


## Handling images of different sizes

Most neural networks expect a fixed-size input, and ours is no exception. But as we've seen above, our images are not all the same size. A common approach is to crop and pad images to a selected aspect ratio, but then we have to make sure we don't cut off parts of the traffic signs in the process. Let's do a simpler thing instead: resize the images to a fixed size and ignore the distortion caused by the different aspect ratios. A person can easily recognize a traffic sign even if it's compressed or stretched a bit, so we hope our model can too.

And while we're at it, let's make the images smaller. The larger the input data, the larger the model, and the slower it is to train. In the early stages of development we want fast training to avoid long waits between iterations.

What are the sizes of our images anyway?


In [ ]:
for image in images[:5]:
    print("shape: {0}, min: {1}, max: {2}".format(image.shape, image.min(), image.max()))


The sizes seem to hover around 128x128. If we resize them to, say, 32x32, we'll reduce the data and model size by a factor of 16, while staying big enough to recognize the signs.

It's a good habit to frequently print min() and max() values — a simple way to verify the range of your data and catch bugs early.


In [ ]:
# Resize images
images32 = [skimage.transform.resize(image, (32, 32), mode='constant', anti_aliasing=True)
            for image in images]
display_images_and_labels(images32, labels)


The 32x32 images are not as sharp but still recognizable. Note that the display above shows the images larger than their real size because matplotlib stretches them to fit the grid. Let's print the sizes of a few images to verify we got it right.


In [ ]:
for image in images32[:5]:
    print("shape: {0}, min: {1}, max: {2}".format(image.shape, image.min(), image.max()))


The sizes are correct. But check the min and max values! They now range from 0 to 1.0, which is different from the 0-255 range we saw above. The resize function did that normalization for us. Remember to multiply by 255 if you ever want to convert images back to the 0-255 range.


# Build the Model (TensorFlow 2 / Keras)

**FIX:** the original tutorial built this model with TensorFlow 0.11 APIs — `tf.placeholder`, `tf.Session`, `tf.contrib.layers.flatten`, `tf.contrib.layers.fully_connected`, and `tf.train.AdamOptimizer` — none of which exist in TensorFlow 2. The cells below rebuild the exact same single-layer network (flatten -> fully-connected -> 62 classes) using `tf.keras`, which runs out of the box on any current TensorFlow installation.


In [ ]:
labels_a = np.array(labels)
images_a = np.array(images32, dtype=np.float32)
print("labels: ", labels_a.shape, "\nimages: ", images_a.shape)


In [ ]:
# A minimum-viable model: flatten the image and feed it into a single
# fully-connected layer with one output per traffic-sign class.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(62, activation='relu'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

model.summary()


## Training

**FIX:** the original code manually created a `tf.Session`, ran an init op, and looped 201 times calling `session.run([train, loss], ...)` on the full dataset each time. With Keras, this entire dance collapses into a single `model.fit(...)` call, which also gives us live loss/accuracy reporting for free.


In [ ]:
history = model.fit(images_a, labels_a, epochs=200, batch_size=len(images_a), verbose=2)


## Using the Model

The `model` object now holds the trained weights. Let's run it on a handful of random images and compare predictions to ground truth.


In [ ]:
# Pick 10 random images
sample_indexes = random.sample(range(len(images32)), 10)
sample_images = np.array([images32[i] for i in sample_indexes], dtype=np.float32)
sample_labels = [labels[i] for i in sample_indexes]

# FIX: session.run([predicted_labels], feed_dict=...) -> model.predict(...)
predictions = model.predict(sample_images)
predicted = np.argmax(predictions, axis=1)
print("Truth:      ", sample_labels)
print("Prediction: ", predicted.tolist())


In [ ]:
# Display the predictions and the ground truth visually.
fig = plt.figure(figsize=(10, 10))
for i in range(len(sample_images)):
    truth = sample_labels[i]
    prediction = predicted[i]
    plt.subplot(5, 2, 1 + i)
    plt.axis('off')
    color = 'green' if truth == prediction else 'red'
    plt.text(40, 10, "Truth:        {0}\nPrediction: {1}".format(truth, prediction),
             fontsize=12, color=color)
    plt.imshow(sample_images[i])


## Evaluation

It's fun to visualize the results, but we need a more precise way to measure accuracy — on images the model hasn't seen during training. That's where the test (validation) set comes in.


In [ ]:
# Load the test dataset.
test_images, test_labels = load_data(test_data_dir)


In [ ]:
# Transform the images, just like we did with the training set.
test_images32 = [skimage.transform.resize(image, (32, 32), mode='constant', anti_aliasing=True)
                 for image in test_images]
display_images_and_labels(test_images32, test_labels)


In [ ]:
# Run predictions against the full test set and compute accuracy.
# FIX: replaced the manual session.run + match-counting loop with
# model.evaluate(...), which does the same computation natively.
test_images32_a = np.array(test_images32, dtype=np.float32)
test_labels_a = np.array(test_labels)

test_loss, test_accuracy = model.evaluate(test_images32_a, test_labels_a, verbose=0)
print("Accuracy: {:.3f}".format(test_accuracy))


**Note:** the original notebook ended with `session.close()` to free the `tf.Session`. TensorFlow 2 runs eagerly and has no explicit session to close, so that step is no longer needed — the `model` object and its weights simply stay in memory for as long as the Python process / kernel is alive.


## Acknowledgements & Source

* This notebook is a modernized, bug-fixed, and renamed (**PanneauNet**) derivative of the original *Traffic Signs Recognition with TensorFlow* tutorial by Waleed Abdulla: https://github.com/waleedka/traffic-signs-tensorflow
* Dataset: **BelgiumTSC — Belgian Traffic Sign Dataset for Classification**, source: http://btsd.ethz.ch/shareddata/

See the project `README.md` for the full list of fixes made to the original code, plus setup instructions for running this notebook locally or via Docker.
